# AIC 2026 — QA/KIS integration & benchmark notebook

This notebook is the canonical QA validation flow for the merged package.

It validates, in order:
1. Repository-root and package imports.
2. Milvus preflight and KIS construction.
3. Agent Core → QA semantic analysis.
4. KIS retrieval → frame evidence → VLM → validation/ranking.
5. A 50-query bilingual benchmark with category-level scoring.
6. Diagnostics for import, retrieval, VLM, ranking, and output failures.

The benchmark's existing 50 ground-truth cases are preserved. They cover COUNT, OBJECT, and SPATIAL questions across English and Vietnamese.

**Important:** the notebook does not invent ground-truth content. Reference answers/frames are used only for evaluation after the model result is produced.


In [ ]:
from pathlib import Path
import json
import os
import sys
import subprocess
from collections import Counter, defaultdict
from pprint import pprint

# Resolve the repository root independently of the notebook working directory.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "qa" / "pipeline.py").exists()
    and (PROJECT_ROOT / "KISBranche").exists()
    and (PROJECT_ROOT / "agent_core" / "src").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (
    (PROJECT_ROOT / "qa" / "pipeline.py").exists()
    and (PROJECT_ROOT / "KISBranche").exists()
):
    raise RuntimeError(
        f"Cannot locate QA project root from {HERE}. "
        "Open the notebook inside QA_KIS_INTEGRATION_FIXED_FULL/qa."
    )

for p in (PROJECT_ROOT, PROJECT_ROOT / "agent_core" / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

os.environ.setdefault("QA_PROJECT_ROOT", str(PROJECT_ROOT))
os.environ.setdefault("QA_DATA_DIR", str(PROJECT_ROOT / "data"))
os.environ.setdefault("QA_KEYFRAMES_DIR", str(PROJECT_ROOT / "data"))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("Python       =", sys.executable)
print("Working dir  =", Path.cwd().resolve())


In [ ]:
# Import smoke test: fail early before constructing heavyweight models.
from qa import QASystem
from qa.agent_core_adapter import AgentCoreAdapter
from qa.milvus_preflight import ensure_milvus_available, is_milvus_available
from qa.qa_query_planner import plan_question
from qa.qa_output import format_submission
from qa.qa_answer_validation import validate_answer
from qa import qa_config

print("qa.QASystem                 : OK")
print("qa.agent_core_adapter       : OK")
print("qa.milvus_preflight         : OK")
print("qa.qa_query_planner         : OK")
print("qa.qa_output                : OK")
print("qa.qa_answer_validation     : OK")
print("Agent Core src              :", PROJECT_ROOT / "agent_core" / "src")

# Agent Core configuration preflight. The .env file is resolved from agent_core/
# by the package itself, independent of the notebook working directory.
try:
    _ac = AgentCoreAdapter().router
    _key = bool(getattr(getattr(_ac, "settings", None), "google_api_key", None))
    print("Agent Core router       : OK")
    print("Agent Core API key      :", "SET" if _key else "MISSING")
    if not _key:
        print("[ACTION] Put GOOGLE_API_KEY in qa_merged/agent_core/.env and restart the kernel.")
except Exception as _exc:
    print("Agent Core preflight    : FAIL")
    print("Agent Core error        :", type(_exc).__name__, str(_exc))


## Runtime preflight

`QASystem.from_shared_kis()` requires Milvus because KIS is the retrieval backbone.

The preflight first checks the configured endpoint, then can start a known Milvus Docker container / compose service according to the `QA_MILVUS_AUTO_START` and `QA_MILVUS_PROMPT` settings.


In [ ]:
# Make the Milvus contract explicit in the notebook instead of discovering it
# only after a deep KIS import fails.
MILVUS_RESULT = ensure_milvus_available()

print("Milvus available :", MILVUS_RESULT.available)
print("Started container:", MILVUS_RESULT.started_container)
print("Container name  :", MILVUS_RESULT.container_name)
print("Message          :", MILVUS_RESULT.message)

LIVE_BACKEND_READY = bool(MILVUS_RESULT.available)
if not LIVE_BACKEND_READY:
    print(
        "\n[WARNING] Live KIS/VLM benchmark is disabled until Milvus is available. "
        "The remaining import/planner/regression tests can still run."
    )


In [ ]:
# KIS + QA construction. Heavy VLM loading remains lazy until the first live QA call.
qa = None
if LIVE_BACKEND_READY:
    qa = QASystem.from_shared_kis()
    print("QASystem          : READY")
    print("KIS provider      :", type(qa.kis).__name__)
    print("VLM               : lazy (loads on first QA call)")
    print("Object metadata   :", type(qa.qa_object_lookup).__name__)
    print("Config DATA_DIR   :", qa_config.DATA_DIR)
    print("Config KEYFRAMES  :", qa_config.KEYFRAMES_DIR)
else:
    print("QASystem          : NOT CONSTRUCTED (Milvus unavailable)")


## Structured input contract

The internal QA pipeline accepts a structured query dictionary at the KIS boundary. For the live benchmark below, the notebook keeps the public input bilingual but lets Agent Core build the English retrieval signal before KIS.


In [ ]:
def build_live_input(question: str, video_query: str | None = None) -> dict:
    """Create an explicit structured request for debugging/inspection."""
    event_text = (video_query or question).strip()
    return {
        "raw_query": event_text,
        "question": question.strip(),
        "input_language": "vi" if any(ch in question.lower() for ch in "ăâđêôơưáàảãạắằẳẵặ") else "en",
        "mode": "QA",
    }

sample_input = build_live_input(
    "Trong trường quay truyền hình, có bao nhiêu người đang đứng phía trước màn hình TV?"
)
pprint(sample_input)


In [ ]:
# Semantic smoke test on both languages.
# Agent Core may require a configured API key; local planner validation still runs offline.
agent_core = AgentCoreAdapter()
semantic_cases = [
    "How many people are standing in front of the TV screen in the television studio?",
    "Trong trường quay truyền hình, có bao nhiêu người đang đứng phía trước màn hình TV?",
    "What is the presenter holding in the television studio?",
    "Trong cảnh ven đường, biển cảnh báo nằm ở đâu?",
]

for q in semantic_cases:
    plan = plan_question(q)
    print("\nQUESTION:", q)
    print("Local target     :", plan.get("target"))
    print("Local expected   :", plan.get("expected_answer_type"))
    print("Relations        :", plan.get("relations"))
    try:
        data = agent_core.analyze_question(q)
        print("Agent answer_type:", data.get("answer_type"))
        print("Agent variants  :", data.get("query_variants"))
    except Exception as exc:
        print("Agent Core       : unavailable/fallback ->", type(exc).__name__, str(exc))


In [ ]:
CASES = [{'id': 'T01', 'question': 'How many people are standing in front of the TV screen in the television studio?', 'reference': {'video_id': 'L21_V001', 'frame_id': 921, 'timestamp_sec': 30.7, 'answer': '2'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T02', 'question': 'In the television studio, how many people are standing directly in front of the TV screen?', 'reference': {'video_id': 'L21_V001', 'frame_id': 921, 'timestamp_sec': 30.7, 'answer': '2'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T03', 'question': 'Looking at the TV screen area in the television studio, how many people are standing in front of it?', 'reference': {'video_id': 'L21_V001', 'frame_id': 921, 'timestamp_sec': 30.7, 'answer': '2'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T04', 'question': 'Trong trường quay truyền hình, có bao nhiêu người đang đứng phía trước màn hình TV?', 'reference': {'video_id': 'L21_V001', 'frame_id': 921, 'timestamp_sec': 30.7, 'answer': '2'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T05', 'question': 'Ở khu vực màn hình TV trong trường quay, có bao nhiêu người đang đứng ngay phía trước?', 'reference': {'video_id': 'L21_V001', 'frame_id': 921, 'timestamp_sec': 30.7, 'answer': '2'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T06', 'question': 'Around the table with documents in front of them, how many people are sitting there?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15146, 'timestamp_sec': 504.867, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T07', 'question': 'In the scene with a table and documents, how many people are seated around the table?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15146, 'timestamp_sec': 504.867, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T08', 'question': 'Looking at the people seated around the table with papers in front of them, how many are there?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15146, 'timestamp_sec': 504.867, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T09', 'question': 'Trong cảnh mọi người ngồi quanh bàn với tài liệu trước mặt, có bao nhiêu người?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15146, 'timestamp_sec': 504.867, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T10', 'question': 'Ở chiếc bàn có các tài liệu đặt trước mặt, có bao nhiêu người đang ngồi quanh bàn?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15146, 'timestamp_sec': 504.867, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T11', 'question': 'In front of the garage, how many motorcycles are parked in a row?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15330, 'timestamp_sec': 511.0, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T12', 'question': 'Looking at the row of motorcycles in front of the garage, how many motorcycles are there?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15330, 'timestamp_sec': 511.0, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T13', 'question': 'At the garage entrance, how many motorcycles can be seen parked side by side?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15330, 'timestamp_sec': 511.0, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T14', 'question': 'Trước gara, có bao nhiêu xe máy đang được đỗ thành một hàng?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15330, 'timestamp_sec': 511.0, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T15', 'question': 'Nhìn vào hàng xe máy phía trước gara, có tất cả bao nhiêu chiếc?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15330, 'timestamp_sec': 511.0, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T16', 'question': 'In the flooded field, how many buffalo are standing together?', 'reference': {'video_id': 'L21_V001', 'frame_id': 12278, 'timestamp_sec': 409.267, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T17', 'question': 'Looking across the flooded field, how many buffalo can be seen standing there?', 'reference': {'video_id': 'L21_V001', 'frame_id': 12278, 'timestamp_sec': 409.267, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T18', 'question': 'At the flooded field scene, how many buffalo are visible?', 'reference': {'video_id': 'L21_V001', 'frame_id': 12278, 'timestamp_sec': 409.267, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T19', 'question': 'Trong cánh đồng bị ngập nước, có bao nhiêu con trâu đang đứng?', 'reference': {'video_id': 'L21_V001', 'frame_id': 12278, 'timestamp_sec': 409.267, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T20', 'question': 'Ở cảnh cánh đồng ngập nước, có tất cả bao nhiêu con trâu có thể nhìn thấy?', 'reference': {'video_id': 'L21_V001', 'frame_id': 12278, 'timestamp_sec': 409.267, 'answer': '4'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T21', 'question': 'In the outdoor walking scene, how many dogs are being walked on leashes?', 'reference': {'video_id': 'L21_V001', 'frame_id': 34032, 'timestamp_sec': 1134.4, 'answer': '2'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T22', 'question': 'Looking at the people walking dogs on leashes, how many dogs are being walked?', 'reference': {'video_id': 'L21_V001', 'frame_id': 34032, 'timestamp_sec': 1134.4, 'answer': '2'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T23', 'question': 'In the outdoor leash-walking scene, how many dogs are visible being walked on leashes?', 'reference': {'video_id': 'L21_V001', 'frame_id': 34032, 'timestamp_sec': 1134.4, 'answer': '2'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T24', 'question': 'Trong cảnh đi bộ ngoài trời, có bao nhiêu con chó đang được dắt bằng dây?', 'reference': {'video_id': 'L21_V001', 'frame_id': 34032, 'timestamp_sec': 1134.4, 'answer': '2'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T25', 'question': 'Ở cảnh mọi người dắt chó đi bộ, có tất cả bao nhiêu con chó?', 'reference': {'video_id': 'L21_V001', 'frame_id': 34032, 'timestamp_sec': 1134.4, 'answer': '2'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T26', 'question': 'In front of the blue backdrop, how many people are standing together and holding papers?', 'reference': {'video_id': 'L21_V001', 'frame_id': 14564, 'timestamp_sec': 485.467, 'answer': '3'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T27', 'question': 'Looking at the group in front of the blue backdrop, how many people are holding papers?', 'reference': {'video_id': 'L21_V001', 'frame_id': 14564, 'timestamp_sec': 485.467, 'answer': '3'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T28', 'question': 'Trước phông nền màu xanh, có bao nhiêu người đang đứng cùng nhau và cầm giấy?', 'reference': {'video_id': 'L21_V001', 'frame_id': 14564, 'timestamp_sec': 485.467, 'answer': '3'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T29', 'question': 'Ở cảnh trước phông nền màu xanh, có bao nhiêu người đang cầm giấy?', 'reference': {'video_id': 'L21_V001', 'frame_id': 14564, 'timestamp_sec': 485.467, 'answer': '3'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T30', 'question': 'Trong cảnh có phông nền xanh, có bao nhiêu người đang đứng cạnh nhau và cầm giấy trên tay?', 'reference': {'video_id': 'L21_V001', 'frame_id': 14564, 'timestamp_sec': 485.467, 'answer': '3'}, 'top_k': 5, 'category': 'COUNT'}, {'id': 'T31', 'question': 'In front of the blue backdrop, what is the woman standing in the middle holding?', 'reference': {'video_id': 'L21_V001', 'frame_id': 14124, 'timestamp_sec': 470.8, 'answer': 'Giấy'}, 'top_k': 5, 'category': 'OBJECT'}, {'id': 'T32', 'question': 'Looking at the woman in the middle of the blue-backdrop scene, what is she holding?', 'reference': {'video_id': 'L21_V001', 'frame_id': 14124, 'timestamp_sec': 470.8, 'answer': 'Giấy'}, 'top_k': 5, 'category': 'OBJECT'}, {'id': 'T33', 'question': 'Trước phông nền màu xanh, người phụ nữ đứng ở giữa đang cầm gì?', 'reference': {'video_id': 'L21_V001', 'frame_id': 14124, 'timestamp_sec': 470.8, 'answer': 'Giấy'}, 'top_k': 5, 'category': 'OBJECT'}, {'id': 'T34', 'question': 'Ở cảnh trước phông nền màu xanh, người phụ nữ ở giữa đang cầm vật gì?', 'reference': {'video_id': 'L21_V001', 'frame_id': 14124, 'timestamp_sec': 470.8, 'answer': 'Giấy'}, 'top_k': 5, 'category': 'OBJECT'}, {'id': 'T35', 'question': 'Trong cảnh phông nền xanh, người phụ nữ đứng ở giữa đang giữ thứ gì?', 'reference': {'video_id': 'L21_V001', 'frame_id': 14124, 'timestamp_sec': 470.8, 'answer': 'Giấy'}, 'top_k': 5, 'category': 'OBJECT'}, {'id': 'T36', 'question': 'In the roadside scene, where is the warning sign located?', 'reference': {'video_id': 'L21_V001', 'frame_id': 2217, 'timestamp_sec': 73.9, 'answer': 'Trên cột'}, 'top_k': 5, 'category': 'SPATIAL'}, {'id': 'T37', 'question': 'Looking at the warning sign near the road, where is the sign mounted?', 'reference': {'video_id': 'L21_V001', 'frame_id': 2217, 'timestamp_sec': 73.9, 'answer': 'Trên cột'}, 'top_k': 5, 'category': 'SPATIAL'}, {'id': 'T38', 'question': 'Trong cảnh ven đường, biển cảnh báo nằm ở đâu?', 'reference': {'video_id': 'L21_V001', 'frame_id': 2217, 'timestamp_sec': 73.9, 'answer': 'Trên cột'}, 'top_k': 5, 'category': 'SPATIAL'}, {'id': 'T39', 'question': 'Ở cảnh có biển cảnh báo, biển báo được đặt ở đâu?', 'reference': {'video_id': 'L21_V001', 'frame_id': 2217, 'timestamp_sec': 73.9, 'answer': 'Trên cột'}, 'top_k': 5, 'category': 'SPATIAL'}, {'id': 'T40', 'question': 'Trong cảnh ven đường có biển cảnh báo, biển báo được gắn ở vị trí nào?', 'reference': {'video_id': 'L21_V001', 'frame_id': 2217, 'timestamp_sec': 73.9, 'answer': 'Trên cột'}, 'top_k': 5, 'category': 'SPATIAL'}, {'id': 'T41', 'question': 'At the speech event, where are the people standing relative to the podium?', 'reference': {'video_id': 'L21_V001', 'frame_id': 23235, 'timestamp_sec': 774.5, 'answer': 'Trước bục phát biểu'}, 'top_k': 5, 'category': 'SPATIAL'}, {'id': 'T42', 'question': 'In the scene with the podium, where are the people standing?', 'reference': {'video_id': 'L21_V001', 'frame_id': 23235, 'timestamp_sec': 774.5, 'answer': 'Trước bục phát biểu'}, 'top_k': 5, 'category': 'SPATIAL'}, {'id': 'T43', 'question': 'Trong buổi phát biểu, những người đó đang đứng ở đâu so với bục phát biểu?', 'reference': {'video_id': 'L21_V001', 'frame_id': 23235, 'timestamp_sec': 774.5, 'answer': 'Trước bục phát biểu'}, 'top_k': 5, 'category': 'SPATIAL'}, {'id': 'T44', 'question': 'Ở cảnh có bục phát biểu, mọi người đang đứng ở vị trí nào?', 'reference': {'video_id': 'L21_V001', 'frame_id': 23235, 'timestamp_sec': 774.5, 'answer': 'Trước bục phát biểu'}, 'top_k': 5, 'category': 'SPATIAL'}, {'id': 'T45', 'question': 'Trong cảnh diễn ra buổi phát biểu, những người đó đứng ở phía nào của bục phát biểu?', 'reference': {'video_id': 'L21_V001', 'frame_id': 23235, 'timestamp_sec': 774.5, 'answer': 'Trước bục phát biểu'}, 'top_k': 5, 'category': 'SPATIAL'}, {'id': 'T46', 'question': 'In the television studio, what is the presenter holding in their hand?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15566, 'timestamp_sec': 518.867, 'answer': 'Một cuốn sách đỏ'}, 'top_k': 5, 'category': 'OBJECT'}, {'id': 'T47', 'question': 'Looking at the presenter in the television studio, what object are they holding?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15566, 'timestamp_sec': 518.867, 'answer': 'Một cuốn sách đỏ'}, 'top_k': 5, 'category': 'OBJECT'}, {'id': 'T48', 'question': 'Trong trường quay truyền hình, người dẫn chương trình đang cầm gì trên tay?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15566, 'timestamp_sec': 518.867, 'answer': 'Một cuốn sách đỏ'}, 'top_k': 5, 'category': 'OBJECT'}, {'id': 'T49', 'question': 'Ở cảnh trường quay, người dẫn chương trình đang cầm vật gì?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15566, 'timestamp_sec': 518.867, 'answer': 'Một cuốn sách đỏ'}, 'top_k': 5, 'category': 'OBJECT'}, {'id': 'T50', 'question': 'Trong cảnh trường quay truyền hình, người dẫn chương trình đang cầm cuốn gì trên tay?', 'reference': {'video_id': 'L21_V001', 'frame_id': 15566, 'timestamp_sec': 518.867, 'answer': 'Một cuốn sách đỏ'}, 'top_k': 5, 'category': 'OBJECT'}]

assert len(CASES) == 50, f"Expected 50 benchmark cases, got {len(CASES)}"
assert {c["id"] for c in CASES} == {f"T{i:02d}" for i in range(1, 51)}

print("Benchmark cases:", len(CASES))
print("By category    :", Counter(c["category"] for c in CASES))


In [ ]:
def _norm(text):
    return " ".join(str(text or "").strip().lower().split())

def evaluate_case(case, prediction):
    ref = case["reference"]
    if not prediction:
        return {
            "id": case["id"], "category": case["category"],
            "answered": False, "answer_correct": False,
            "video_correct": False, "frame_tolerant": False,
            "prediction": None, "reference": ref,
        }

    p = prediction
    gt_answer = _norm(ref.get("answer"))
    pred_answer = _norm(p.get("answer"))
    gt_video = str(ref.get("video_id"))
    pred_video = str(p.get("video_id"))
    try:
        gt_frame = int(ref.get("frame_id"))
        pred_frame = int(p.get("frame_id"))
        frame_ok = abs(gt_frame - pred_frame) <= 3
    except (TypeError, ValueError):
        frame_ok = False

    return {
        "id": case["id"], "category": case["category"],
        "answered": True,
        "answer_correct": pred_answer == gt_answer,
        "video_correct": pred_video == gt_video,
        "frame_tolerant": frame_ok,
        "prediction": p, "reference": ref,
    }

def summarize_evaluation(results):
    total = len(results)
    answered = sum(r["answered"] for r in results)
    answer = sum(r["answer_correct"] for r in results)
    video = sum(r["video_correct"] for r in results)
    frame = sum(r["frame_tolerant"] for r in results)

    by_cat = {}
    for cat in sorted({r["category"] for r in results}):
        rows = [r for r in results if r["category"] == cat]
        n = len(rows)
        by_cat[cat] = {
            "n": n,
            "coverage": sum(r["answered"] for r in rows) / n if n else 0.0,
            "answer_accuracy": sum(r["answer_correct"] for r in rows) / n if n else 0.0,
            "video_recall": sum(r["video_correct"] for r in rows) / n if n else 0.0,
            "frame_tolerant": sum(r["frame_tolerant"] for r in rows) / n if n else 0.0,
        }

    return {
        "n": total,
        "coverage": answered / total if total else 0.0,
        "answer_accuracy": answer / total if total else 0.0,
        "video_recall": video / total if total else 0.0,
        "frame_tolerant": frame / total if total else 0.0,
        "by_category": by_cat,
    }


## Live benchmark

Set `RUN_LIVE_BENCHMARK = True` to run all 50 cases through the actual pipeline.

The pipeline is intentionally evaluated with `top_k=5` so retrieval quality and VLM selection are tested without turning the benchmark into 50×100 VLM calls.


In [ ]:
RUN_LIVE_BENCHMARK = False
TOP_K = 5

live_results = []
if RUN_LIVE_BENCHMARK and not LIVE_BACKEND_READY:
    raise RuntimeError("RUN_LIVE_BENCHMARK=True but Milvus/KIS backend is not ready.")
elif RUN_LIVE_BENCHMARK:
    for idx, case in enumerate(CASES, 1):
        question = case["question"]
        print(f"\n{'=' * 100}\n[{idx:02d}/50] {case['id']} | {case['category']}\nQUESTION: {question}\n{'=' * 100}")
        try:
            rows = qa.query(question, top_k=TOP_K)
            pred = rows[0] if rows else None
            result = evaluate_case(case, pred)
            result["diagnostics"] = list(qa.last_run_diagnostics)
            live_results.append(result)
            print("PREDICTION:", pred)
            print("MATCH:", {k: result[k] for k in ("answer_correct","video_correct","frame_tolerant")})
        except Exception as exc:
            live_results.append({
                "id": case["id"], "category": case["category"],
                "answered": False, "answer_correct": False,
                "video_correct": False, "frame_tolerant": False,
                "prediction": None, "reference": case["reference"],
                "error": repr(exc), "diagnostics": list(qa.last_run_diagnostics),
            })
            print("ERROR:", repr(exc))
else:
    print("RUN_LIVE_BENCHMARK=False -> skipped live model calls.")
    print("Set RUN_LIVE_BENCHMARK=True only after the Milvus preflight reports ready.")


In [ ]:
if live_results:
    summary = summarize_evaluation(live_results)
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    failures = [
        r for r in live_results
        if not (r["answer_correct"] and r["video_correct"] and r["frame_tolerant"])
    ]
    print("\nFailures:", len(failures))
    for r in failures[:20]:
        print(r["id"], r["category"], r.get("error"), r.get("prediction"))
else:
    print("No live benchmark results to score.")


## Legacy Query Test (kept for direct comparison)

This section restores the original query-by-query test flow. The 50 original test cells are kept individually so you can compare outputs against earlier QA runs.

**Important:** this is intentionally separate from the new benchmark loop. It is a compatibility/manual-evaluation section, not a replacement for the benchmark metrics.


In [ ]:
from qa.qa_query_planner import plan_question
from qa.qa_output import format_submission
print("Legacy query-test helpers: OK")


In [ ]:
# Legacy-compatible runner. The old notebook exposed decompose/retrieval variants;
# the repaired planner now exposes the equivalent structured plan directly.
def run_legacy_case(case_id, language, question, reference, top_k=5):
    plan = plan_question(question)
    print("=" * 110)
    print(f"{case_id} | {language}")
    print("QUESTION          :", question)
    print("KIS QUERY         :", plan.get("normalized", question))
    print("KIS VARIANTS      :", {"target": plan.get("target"), "target_terms": plan.get("target_terms"), "relations": plan.get("relations")})
    print("QA QUERY           :", plan.get("target") or question)
    print("REFERENCE         :", reference)
    print("PLAN TYPE         :", plan.get("expected_answer_type"), "| operation:", plan.get("operation"), "| complexity:", plan.get("complexity"))
    print("-" * 110)
    if qa is None:
        print("MODEL: SKIPPED — live QASystem is not ready (Milvus/backend unavailable).")
        return None
    rows = qa.query(question, top_k=top_k)
    if not rows:
        print("MODEL: NO RESULT")
    else:
        print("MODEL:", rows[0])
        print("SUBMISSION:", format_submission(rows))
    return rows


In [ ]:
def format_mmss(seconds):
    if seconds is None:
        return "--:--"
    try:
        total = max(0, int(round(float(seconds))))
    except (TypeError, ValueError):
        return "--:--"
    minutes, secs = divmod(total, 60)
    return f"{minutes:02d}:{secs:02d}"


def run_case(case_id: str, language: str, question: str, reference: dict | None = None, top_k: int = 5):
    """Run one T01-T50 query and print only compact Top-K results.

    Ground-truth reference is evaluation-only and is never passed to qa.query().
    Official submission rows remain video_id,frame_id,answer; timestamp is debug display only.
    """
    if qa is None:
        raise RuntimeError("QASystem is not ready. Run the construction cell first.")
    question = str(question or "").strip()
    if not question:
        raise ValueError("question must be non-empty")

    rows = qa.query(question, top_k=int(top_k))
    print(f"{case_id} | {language} | Top-{len(rows)}")
    if not rows:
        print("NO RESULT")
        return {"prediction": None, "rows": [], "diagnostics": list(getattr(qa, "last_run_diagnostics", []))}

    for rank, row in enumerate(rows, 1):
        video_id = row.get("video_id", "")
        frame_id = row.get("frame_id", "")
        answer = str(row.get("answer", "")).strip()
        try:
            ts = qa.frame_loader.timestamp_sec(video_id, int(frame_id))
        except Exception:
            ts = None
        print(f"#{rank} {video_id} | frame={frame_id} | time={format_mmss(ts)} | answer={answer}")

    # Official submission stays exactly video_id,frame_id,answer.
    submission = format_submission(rows)
    result = {
        "prediction": rows[0],
        "rows": rows,
        "submission": submission,
        "diagnostics": list(getattr(qa, "last_run_diagnostics", [])),
    }
    if reference is not None:
        result["evaluation"] = evaluate_case(
            {"id": case_id, "category": "INTERACTIVE", "reference": reference},
            rows[0],
        )
    return result


In [ ]:
# T01 | EN | COUNT | grounded scene Q001
QUESTION = 'How many people are standing in front of the TV screen in the television studio?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 921, 'timestamp_sec': 30.7, 'answer': '2'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T01", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T02 | EN | COUNT | grounded scene Q001
QUESTION = 'In the television studio, how many people are standing directly in front of the TV screen?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 921, 'timestamp_sec': 30.7, 'answer': '2'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T02", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T03 | EN | COUNT | grounded scene Q001
QUESTION = 'Looking at the TV screen area in the television studio, how many people are standing in front of it?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 921, 'timestamp_sec': 30.7, 'answer': '2'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T03", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T04 | VI | COUNT | grounded scene Q001
QUESTION = 'Trong trường quay truyền hình, có bao nhiêu người đang đứng phía trước màn hình TV?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 921, 'timestamp_sec': 30.7, 'answer': '2'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T04", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T05 | VI | COUNT | grounded scene Q001
QUESTION = 'Ở khu vực màn hình TV trong trường quay, có bao nhiêu người đang đứng ngay phía trước?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 921, 'timestamp_sec': 30.7, 'answer': '2'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T05", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T06 | EN | COUNT | grounded scene Q002
QUESTION = 'Around the table with documents in front of them, how many people are sitting there?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15146, 'timestamp_sec': 504.867, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T06", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T07 | EN | COUNT | grounded scene Q002
QUESTION = 'In the scene with a table and documents, how many people are seated around the table?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15146, 'timestamp_sec': 504.867, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T07", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T08 | EN | COUNT | grounded scene Q002
QUESTION = 'Looking at the people seated around the table with papers in front of them, how many are there?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15146, 'timestamp_sec': 504.867, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T08", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T09 | VI | COUNT | grounded scene Q002
QUESTION = 'Trong cảnh mọi người ngồi quanh bàn với tài liệu trước mặt, có bao nhiêu người?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15146, 'timestamp_sec': 504.867, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T09", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T10 | VI | COUNT | grounded scene Q002
QUESTION = 'Ở chiếc bàn có các tài liệu đặt trước mặt, có bao nhiêu người đang ngồi quanh bàn?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15146, 'timestamp_sec': 504.867, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T10", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T11 | EN | COUNT | grounded scene Q003
QUESTION = 'In front of the garage, how many motorcycles are parked in a row?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15330, 'timestamp_sec': 511.0, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T11", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T12 | EN | COUNT | grounded scene Q003
QUESTION = 'Looking at the row of motorcycles in front of the garage, how many motorcycles are there?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15330, 'timestamp_sec': 511.0, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T12", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T13 | EN | COUNT | grounded scene Q003
QUESTION = 'At the garage entrance, how many motorcycles can be seen parked side by side?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15330, 'timestamp_sec': 511.0, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T13", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T14 | VI | COUNT | grounded scene Q003
QUESTION = 'Trước gara, có bao nhiêu xe máy đang được đỗ thành một hàng?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15330, 'timestamp_sec': 511.0, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T14", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T15 | VI | COUNT | grounded scene Q003
QUESTION = 'Nhìn vào hàng xe máy phía trước gara, có tất cả bao nhiêu chiếc?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15330, 'timestamp_sec': 511.0, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T15", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T16 | EN | COUNT | grounded scene Q004
QUESTION = 'In the flooded field, how many buffalo are standing together?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 12278, 'timestamp_sec': 409.267, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T16", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T17 | EN | COUNT | grounded scene Q004
QUESTION = 'Looking across the flooded field, how many buffalo can be seen standing there?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 12278, 'timestamp_sec': 409.267, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T17", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T18 | EN | COUNT | grounded scene Q004
QUESTION = 'At the flooded field scene, how many buffalo are visible?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 12278, 'timestamp_sec': 409.267, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T18", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T19 | VI | COUNT | grounded scene Q004
QUESTION = 'Trong cánh đồng bị ngập nước, có bao nhiêu con trâu đang đứng?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 12278, 'timestamp_sec': 409.267, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T19", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T20 | VI | COUNT | grounded scene Q004
QUESTION = 'Ở cảnh cánh đồng ngập nước, có tất cả bao nhiêu con trâu có thể nhìn thấy?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 12278, 'timestamp_sec': 409.267, 'answer': '4'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T20", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T21 | EN | COUNT | grounded scene Q005
QUESTION = 'In the outdoor walking scene, how many dogs are being walked on leashes?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 34032, 'timestamp_sec': 1134.4, 'answer': '2'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T21", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T22 | EN | COUNT | grounded scene Q005
QUESTION = 'Looking at the people walking dogs on leashes, how many dogs are being walked?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 34032, 'timestamp_sec': 1134.4, 'answer': '2'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T22", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T23 | EN | COUNT | grounded scene Q005
QUESTION = 'In the outdoor leash-walking scene, how many dogs are visible being walked on leashes?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 34032, 'timestamp_sec': 1134.4, 'answer': '2'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T23", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T24 | VI | COUNT | grounded scene Q005
QUESTION = 'Trong cảnh đi bộ ngoài trời, có bao nhiêu con chó đang được dắt bằng dây?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 34032, 'timestamp_sec': 1134.4, 'answer': '2'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T24", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T25 | VI | COUNT | grounded scene Q005
QUESTION = 'Ở cảnh mọi người dắt chó đi bộ, có tất cả bao nhiêu con chó?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 34032, 'timestamp_sec': 1134.4, 'answer': '2'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T25", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T26 | EN | COUNT | grounded scene Q006
QUESTION = 'In front of the blue backdrop, how many people are standing together and holding papers?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 14564, 'timestamp_sec': 485.467, 'answer': '3'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T26", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T27 | EN | COUNT | grounded scene Q006
QUESTION = 'Looking at the group in front of the blue backdrop, how many people are holding papers?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 14564, 'timestamp_sec': 485.467, 'answer': '3'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T27", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T28 | VI | COUNT | grounded scene Q006
QUESTION = 'Trước phông nền màu xanh, có bao nhiêu người đang đứng cùng nhau và cầm giấy?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 14564, 'timestamp_sec': 485.467, 'answer': '3'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T28", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T29 | VI | COUNT | grounded scene Q006
QUESTION = 'Ở cảnh trước phông nền màu xanh, có bao nhiêu người đang cầm giấy?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 14564, 'timestamp_sec': 485.467, 'answer': '3'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T29", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T30 | VI | COUNT | grounded scene Q006
QUESTION = 'Trong cảnh có phông nền xanh, có bao nhiêu người đang đứng cạnh nhau và cầm giấy trên tay?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 14564, 'timestamp_sec': 485.467, 'answer': '3'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T30", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T31 | EN | OBJECT | grounded scene Q007
QUESTION = 'In front of the blue backdrop, what is the woman standing in the middle holding?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 14124, 'timestamp_sec': 470.8, 'answer': 'Giấy'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T31", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T32 | EN | OBJECT | grounded scene Q007
QUESTION = 'Looking at the woman in the middle of the blue-backdrop scene, what is she holding?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 14124, 'timestamp_sec': 470.8, 'answer': 'Giấy'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T32", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T33 | VI | OBJECT | grounded scene Q007
QUESTION = 'Trước phông nền màu xanh, người phụ nữ đứng ở giữa đang cầm gì?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 14124, 'timestamp_sec': 470.8, 'answer': 'Giấy'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T33", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T34 | VI | OBJECT | grounded scene Q007
QUESTION = 'Ở cảnh trước phông nền màu xanh, người phụ nữ ở giữa đang cầm vật gì?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 14124, 'timestamp_sec': 470.8, 'answer': 'Giấy'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T34", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T35 | VI | OBJECT | grounded scene Q007
QUESTION = 'Trong cảnh phông nền xanh, người phụ nữ đứng ở giữa đang giữ thứ gì?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 14124, 'timestamp_sec': 470.8, 'answer': 'Giấy'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T35", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T36 | EN | SPATIAL | grounded scene Q008
QUESTION = 'In the roadside scene, where is the warning sign located?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 2217, 'timestamp_sec': 73.9, 'answer': 'Trên cột'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T36", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T37 | EN | SPATIAL | grounded scene Q008
QUESTION = 'Looking at the warning sign near the road, where is the sign mounted?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 2217, 'timestamp_sec': 73.9, 'answer': 'Trên cột'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T37", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T38 | VI | SPATIAL | grounded scene Q008
QUESTION = 'Trong cảnh ven đường, biển cảnh báo nằm ở đâu?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 2217, 'timestamp_sec': 73.9, 'answer': 'Trên cột'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T38", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T39 | VI | SPATIAL | grounded scene Q008
QUESTION = 'Ở cảnh có biển cảnh báo, biển báo được đặt ở đâu?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 2217, 'timestamp_sec': 73.9, 'answer': 'Trên cột'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T39", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T40 | VI | SPATIAL | grounded scene Q008
QUESTION = 'Trong cảnh ven đường có biển cảnh báo, biển báo được gắn ở vị trí nào?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 2217, 'timestamp_sec': 73.9, 'answer': 'Trên cột'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T40", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T41 | EN | SPATIAL | grounded scene Q009
QUESTION = 'At the speech event, where are the people standing relative to the podium?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 23235, 'timestamp_sec': 774.5, 'answer': 'Trước bục phát biểu'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T41", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T42 | EN | SPATIAL | grounded scene Q009
QUESTION = 'In the scene with the podium, where are the people standing?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 23235, 'timestamp_sec': 774.5, 'answer': 'Trước bục phát biểu'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T42", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T43 | VI | SPATIAL | grounded scene Q009
QUESTION = 'Trong buổi phát biểu, những người đó đang đứng ở đâu so với bục phát biểu?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 23235, 'timestamp_sec': 774.5, 'answer': 'Trước bục phát biểu'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T43", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T44 | VI | SPATIAL | grounded scene Q009
QUESTION = 'Ở cảnh có bục phát biểu, mọi người đang đứng ở vị trí nào?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 23235, 'timestamp_sec': 774.5, 'answer': 'Trước bục phát biểu'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T44", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T45 | VI | SPATIAL | grounded scene Q009
QUESTION = 'Trong cảnh diễn ra buổi phát biểu, những người đó đứng ở phía nào của bục phát biểu?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 23235, 'timestamp_sec': 774.5, 'answer': 'Trước bục phát biểu'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T45", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T46 | EN | OBJECT | grounded scene Q010
QUESTION = 'In the television studio, what is the presenter holding in their hand?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15566, 'timestamp_sec': 518.867, 'answer': 'Một cuốn sách đỏ'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T46", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T47 | EN | OBJECT | grounded scene Q010
QUESTION = 'Looking at the presenter in the television studio, what object are they holding?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15566, 'timestamp_sec': 518.867, 'answer': 'Một cuốn sách đỏ'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T47", "EN", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T48 | VI | OBJECT | grounded scene Q010
QUESTION = 'Trong trường quay truyền hình, người dẫn chương trình đang cầm gì trên tay?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15566, 'timestamp_sec': 518.867, 'answer': 'Một cuốn sách đỏ'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T48", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T49 | VI | OBJECT | grounded scene Q010
QUESTION = 'Ở cảnh trường quay, người dẫn chương trình đang cầm vật gì?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15566, 'timestamp_sec': 518.867, 'answer': 'Một cuốn sách đỏ'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T49", "VI", QUESTION, REFERENCE, top_k=TOP_K)


In [ ]:
# T50 | VI | OBJECT | grounded scene Q010
QUESTION = 'Trong cảnh trường quay truyền hình, người dẫn chương trình đang cầm cuốn gì trên tay?'
TOP_K = 5
REFERENCE = {'video_id': 'L21_V001', 'frame_id': 15566, 'timestamp_sec': 518.867, 'answer': 'Một cuốn sách đỏ'}
FRAME_ID = REFERENCE["frame_id"]
TIMESTAMP_SEC = REFERENCE["timestamp_sec"]
print(f"REFERENCE FRAME={FRAME_ID} | TIMESTAMP={TIMESTAMP_SEC:.3f}s")

run_case("T50", "VI", QUESTION, REFERENCE, top_k=TOP_K)


### Legacy test interpretation

Use these 50 cells for direct visual comparison with the older notebook. Use the newer benchmark section for aggregate accuracy/coverage and the diagnostics section for pipeline-stage failures.


In [ ]:
print("Legacy query compatibility section restored: 50 individual test cells.")
print("For comparable manual testing, keep TOP_K=5 and do not pass the reference answer into qa.query().")


## Diagnostics by pipeline stage

This section is designed to catch the failure mode hidden by a single final score: a question can be wrong because Agent Core failed, KIS returned poor candidates, evidence was missing, VLM answered incorrectly, or ranking selected the wrong candidate.


In [ ]:
if live_results:
    stage_total = Counter()
    stage_fail = Counter()
    for row in live_results:
        for item in row.get("diagnostics", []):
            stage = item.get("stage", "unknown")
            stage_total[stage] += 1
            if item.get("ok") is False or item.get("error"):
                stage_fail[stage] += 1

    print("Stage events :", dict(stage_total))
    print("Stage fails  :", dict(stage_fail))
else:
    print("Run the live benchmark first to populate stage diagnostics.")


## Reproducible regression test

The notebook also runs the repository test suite with the correct root-level Agent Core import path. A plain `pytest -q` used to fail during collection because `aic_agent_core` was not visible from the merged project root.


In [ ]:
test_cmd = [
    sys.executable, "-m", "pytest", "-q",
    str(PROJECT_ROOT / "qa"),
    str(PROJECT_ROOT / "agent_core" / "tests"),
]
print("Running:", " ".join(test_cmd))
completed = subprocess.run(
    test_cmd,
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"Regression suite failed with exit code {completed.returncode}")
